In [ ]:
from pydantic import BaseModel, Field
from openai import AsyncOpenAI
from typing import List, Literal
from enum import Enum
import json
from pathlib import Path

client = AsyncOpenAI()

In [ ]:
class ProceedSignal(str, Enum):
    """Investment decision signal."""
    STRONG_YES = "strong_yes"
    QUESTIONABLE = "questionable"
    STRONG_NO = "strong_no"


class DealVerdict(BaseModel):
    """Structured verdict on a potential PE deal."""
    
    proceed_signal: ProceedSignal = Field(
        description="Overall investment signal: strong_yes (high conviction opportunity), questionable (needs more analysis), or strong_no (pass on deal)"
    )
    
    evidence_pro: List[str] = Field(
        description="Key positive factors supporting investment: competitive advantages, growth drivers, financial strengths, market opportunities",
        default_factory=list
    )
    
    evidence_contra: List[str] = Field(
        description="Key concerns or risks against investment: competitive threats, financial weaknesses, market headwinds, execution risks",
        default_factory=list
    )
    
    similar_deals: str = Field(
        description="Comparable past PE deals (fictional examples) in similar sectors/situations that provide context. Format as brief descriptions of buyouts, growth equity, add-on acquisitions, etc."
    )
    
    key_information_requested: str = Field(
        description="Critical additional information, analysis, or due diligence needed to make final investment decision, even if signal is positive"
    )


class TrainingExample(BaseModel):
    """Complete training data example for fine-tuning."""
    
    report: str = Field(
        description="The original deal sourcing research report that was analyzed"
    )
    
    reasoning: str = Field(
        description="Step-by-step analytical reasoning that led to the verdict, covering competition, customers, financials, and growth"
    )
    
    verdict: DealVerdict = Field(
        description="The structured investment verdict with proceed signal and supporting evidence"
    )

In [ ]:
async def generate_training_example(
    report_path: str,
    model: str = "gpt-4.1-mini",
    temperature: float = 2.0,
    client: AsyncOpenAI = None,
) -> TrainingExample:
    
    # Read the report
    report_text = Path(report_path).read_text()
    
    system_prompt = """You are a senior private equity analyst evaluating potential investment opportunities.

Your task is to analyze the provided deal sourcing research report and generate a comprehensive investment evaluation.

Think through:
1. COMPETITION: Market position, competitive dynamics, defensibility
2. CUSTOMERS: Customer quality, retention, concentration risks
3. FINANCIALS: Growth trajectory, profitability, cash generation, balance sheet
4. GROWTH OPPORTUNITIES: Organic growth, M&A, operational improvements

Based on this analysis, determine:
- proceed_signal: strong_yes (compelling opportunity), questionable (uncertain/needs work), strong_no (pass)
- evidence_pro: specific positive factors (3-6 points)
- evidence_contra: specific concerns or risks (3-6 points)
- similar_deals: fictional but realistic past PE deals that provide comps (e.g., "Vista Equity's buyout of Marketo in 2016 - SaaS marketing automation with similar growth profile and competitive position")
- key_information_requested: critical additional diligence items needed

Be specific and grounded in the report data. Generate realistic, diverse examples suitable for model fine-tuning."""

    user_prompt = f"""Analyze this deal sourcing report and provide your investment evaluation:

{report_text}

Think step-by-step through Competition, Customers, Financials, and Growth Opportunities, then deliver your verdict."""
    
    completion = await client.beta.chat.completions.parse(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        response_format=TrainingExample,
    )
    
    return completion.choices[0].message.parsed

In [ ]:
async def generate_batch(
    report_path: str,
    num_examples: int = 10,
    client: AsyncOpenAI = None,
    temperature: float = 1.5,
) -> List[TrainingExample]:
    """
    Generate multiple training examples from the same report (with temperature variation).
    
    Args:
        report_path: Path to markdown report file
        num_examples: Number of diverse examples to generate
        client: AsyncOpenAI client instance
        temperature: Base sampling temperature
        
    Returns:
        List of TrainingExample objects
    """
    import asyncio
    
    tasks = [
        generate_training_example(
            report_path=report_path,
            temperature=temperature,
            client=client
        )
        for _ in range(num_examples)
    ]
    
    results = await asyncio.gather(*tasks)
    return results

In [ ]:
def save_training_data(
    examples: List[TrainingExample],
    output_path: str = "training_data.jsonl"
):
    """
    Save training examples to JSONL format for fine-tuning.
    
    Args:
        examples: List of TrainingExample objects
        output_path: Path to output JSONL file
    """
    with open(output_path, 'w') as f:
        for example in examples:
            # Convert to dict and write as JSON line
            f.write(example.model_dump_json() + '\n')
    
    print(f"Saved {len(examples)} training examples to {output_path}")

In [ ]:
batch = await generate_batch(
    report_path="../data/output/Duolingo_2025-11-18_21-47-42.md",
    num_examples=5,
    client=client
)

# Save to file
save_training_data(batch, "../data/training_data_examples.jsonl")